# HCP Engagement Optimizer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Classify engagement levels** using `SNOWFLAKE.ML.CLASSIFICATION`
2. **Engineer engagement features** from multi-channel data
3. **Score physicians** (Alto/Medio/Bajo engagement)
4. **Optimize rep targeting** based on ML predictions
5. **Track engagement trends** over time

---

## What You'll Build

A **physician engagement classifier** that optimizes sales targeting:
- Multi-class classification (Alto/Medio/Bajo engagement)
- Feature engineering from events, samples, emails
- Territory planning optimization
- Rep resource allocation recommendations
- Engagement tracking dashboard

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 18 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `SNOWFLAKE.ML.CLASSIFICATION` - Multi-class ML model
- `!PREDICT()` - Score new physicians
- `COUNT()` with `FILTER` - Conditional aggregations
- `DATEDIFF()` - Calculate recency metrics
- Window functions - Engagement trend analysis

Let's begin!

---

## Paso 1: Configuración del Entorno

### HCP Engagement Optimization

**Goal**: Maximize prescribing by focusing rep efforts on high-propensity HCPs

### Key Metrics

- **Prescribing propensity**: Probability HCP will prescribe
- **Engagement score**: Quality of touchpoints
- **Channel effectiveness**: Which channels drive prescribing
- **Rep productivity**: Targets reached per rep

### Why ML for HCP Engagement?

Predict behavior to:
- Prioritize rep call lists
- Personalize outreach strategies
- Allocate resources efficiently
- Improve ROI on field force

In [ ]:
CREATE DATABASE IF NOT EXISTS HCP_ENGAGEMENT_DB;
CREATE SCHEMA IF NOT EXISTS HCP_ENGAGEMENT_DB.PUBLIC;
USE SCHEMA HCP_ENGAGEMENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`hcps`** - Healthcare provider profiles
2. **`engagements`** - Rep visits, emails, events
3. **`prescriptions`** - Actual prescribing behavior (target variable)

### Features for Prediction

- **Engagement frequency**: # of touchpoints
- **Engagement recency**: Days since last contact
- **Channel diversity**: # of different channels used
- **Response rate**: % of emails opened, meetings accepted
- **Specialty match**: Does HCP specialty align with product?

In [ ]:
-- HCP profiles
CREATE OR REPLACE TABLE hcps (
    hcp_id VARCHAR(50) PRIMARY KEY,
    specialty VARCHAR(100),
    years_in_practice INTEGER,
    patient_volume VARCHAR(20),
    institution_type VARCHAR(50),
    tier VARCHAR(20)  -- Tier 1/2/3 based on potential
);

-- Engagement history
CREATE OR REPLACE TABLE engagements (
    engagement_id VARCHAR(50) PRIMARY KEY,
    hcp_id VARCHAR(50),
    engagement_date DATE,
    engagement_type VARCHAR(50),  -- Rep_Visit, Email, Event, Sample
    engagement_score INTEGER,  -- 1-10 quality score
    rep_id VARCHAR(50)
);

-- Prescribing outcomes
CREATE OR REPLACE TABLE prescriptions (
    prescription_id VARCHAR(50) PRIMARY KEY,
    hcp_id VARCHAR(50),
    prescription_date DATE,
    product_name VARCHAR(100),
    num_patients INTEGER
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos HCP Data

Creating realistic HCP profiles, engagement history, and prescribing data

In [ ]:
-- Generar HCP profiles (500 HCPs)
INSERT INTO hcps
SELECT 
    'HCP' || LPAD(SEQ4(), 4, '0') as hcp_id,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Endocrinology'
        WHEN 2 THEN 'Primary Care'
        WHEN 3 THEN 'Internal Medicine'
        ELSE 'Family Practice'
    END as specialty,
    FLOOR(UNIFORM(2, 35, RANDOM())) as years_in_practice,
    CASE FLOOR(UNIFORM(1, 4, RANDOM()))
        WHEN 1 THEN 'High (200+)'
        WHEN 2 THEN 'Medium (50-200)'
        ELSE 'Low (<50)'
    END as patient_volume,
    CASE FLOOR(UNIFORM(1, 4, RANDOM()))
        WHEN 1 THEN 'Academic Medical Center'
        WHEN 2 THEN 'Community Hospital'
        ELSE 'Private Practice'
    END as institution_type,
    CASE 
        WHEN UNIFORM(0, 1, RANDOM()) > 0.7 THEN 'Tier 1'
        WHEN UNIFORM(0, 1, RANDOM()) > 0.5 THEN 'Tier 2'
        ELSE 'Tier 3'
    END as tier
FROM TABLE(GENERATOR(ROWCOUNT => 500));

-- Generar engagement history (5,000 engagements)
INSERT INTO engagements
SELECT 
    UUID_STRING() as engagement_id,
    'HCP' || LPAD(FLOOR(UNIFORM(1, 501, RANDOM())), 4, '0') as hcp_id,
    DATEADD('day', -FLOOR(UNIFORM(1, 365, RANDOM())), CURRENT_DATE()) as engagement_date,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Rep_Visit'
        WHEN 2 THEN 'Email'
        WHEN 3 THEN 'Event'
        ELSE 'Sample_Drop'
    END as engagement_type,
    FLOOR(UNIFORM(3, 11, RANDOM())) as engagement_score,
    'REP' || LPAD(FLOOR(UNIFORM(1, 51, RANDOM())), 3, '0') as rep_id
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

-- Generar prescriptions (HCPs with high engagement more likely to prescribe)
INSERT INTO prescriptions
WITH hcp_engagement_level AS (
    SELECT 
        h.hcp_id,
        h.specialty,
        COUNT(e.engagement_id) as total_engagements
    FROM hcps h
    LEFT JOIN engagements e ON h.hcp_id = e.hcp_id
    GROUP BY h.hcp_id, h.specialty
),
prescribers AS (
    SELECT hcp_id, specialty, total_engagements
    FROM hcp_engagement_level
    -- Altoer engagement = higher chance of prescribing
    WHERE (total_engagements >= 8 AND UNIFORM(0,1,RANDOM()) > 0.3)
       OR (total_engagements BETWEEN 4 AND 7 AND UNIFORM(0,1,RANDOM()) > 0.6)
       OR (total_engagements < 4 AND UNIFORM(0,1,RANDOM()) > 0.9)
)
SELECT 
    UUID_STRING() as prescription_id,
    p.hcp_id,
    DATEADD('day', -FLOOR(UNIFORM(1, 180, RANDOM())), CURRENT_DATE()) as prescription_date,
    'Xarelto' as product_name,
    FLOOR(UNIFORM(1, 20, RANDOM())) as num_patients
FROM prescribers p
CROSS JOIN TABLE(GENERATOR(ROWCOUNT => 2))
WHERE UNIFORM(0,1,RANDOM()) > 0.5;

-- Verify data
SELECT 
    (SELECT COUNT(*) FROM hcps) as hcps,
    (SELECT COUNT(*) FROM engagements) as engagements,
    (SELECT COUNT(*) FROM prescriptions) as prescriptions,
    (SELECT COUNT(DISTINCT hcp_id) FROM prescriptions) as prescribing_hcps;

---

## Paso 4: Engineer Features for ML Model

### Ingeniería de Variables

Create predictive features from engagement history:
- **Engagement frequency**: Total touchpoints in last 90 days
- **Recency**: Days since last engagement
- **Channel diversity**: # of unique engagement types
- **Engagement quality**: Average engagement score
- **Specialty alignment**: Binary flag for target specialties

In [ ]:
-- Engineer features for propensity modeling
CREATE OR REPLACE TABLE hcp_features AS
WITH engagement_stats AS (
    SELECT 
        e.hcp_id,
        COUNT(*) as total_engagements_90d,
        COUNT(DISTINCT e.engagement_type) as channel_diversity,
        AVG(e.engagement_score) as avg_engagement_quality,
        MAX(e.engagement_date) as last_engagement_date,
        DATEDIFF('day', last_engagement_date, CURRENT_DATE()) as days_since_last_engagement,
        COUNT(CASE WHEN e.engagement_type = 'Rep_Visit' THEN 1 END) as rep_visits_90d,
        COUNT(CASE WHEN e.engagement_type = 'Email' THEN 1 END) as emails_90d,
        COUNT(CASE WHEN e.engagement_type = 'Event' THEN 1 END) as events_90d
    FROM engagements e
    WHERE e.engagement_date >= DATEADD('day', -90, CURRENT_DATE())
    GROUP BY e.hcp_id
)
SELECT 
    h.hcp_id,
    -- Demographics
    h.specialty,
    h.years_in_practice,
    CASE h.patient_volume
        WHEN 'High (200+)' THEN 3
        WHEN 'Medium (50-200)' THEN 2
        ELSE 1
    END as patient_volume_score,
    CASE h.institution_type
        WHEN 'Academic Medical Center' THEN 3
        WHEN 'Community Hospital' THEN 2
        ELSE 1
    END as institution_score,
    CASE h.tier
        WHEN 'Tier 1' THEN 3
        WHEN 'Tier 2' THEN 2
        ELSE 1
    END as tier_score,
    -- Engagement features
    COALESCE(es.total_engagements_90d, 0) as total_engagements_90d,
    COALESCE(es.channel_diversity, 0) as channel_diversity,
    COALESCE(es.avg_engagement_quality, 0)::FLOAT as avg_engagement_quality,
    COALESCE(es.days_since_last_engagement, 999) as days_since_last_engagement,
    COALESCE(es.rep_visits_90d, 0) as rep_visits_90d,
    COALESCE(es.emails_90d, 0) as emails_90d,
    COALESCE(es.events_90d, 0) as events_90d,
    -- Target variable
    CASE WHEN p.hcp_id IS NOT NULL THEN 1 ELSE 0 END as is_prescriber
FROM hcps h
LEFT JOIN engagement_stats es ON h.hcp_id = es.hcp_id
LEFT JOIN (SELECT DISTINCT hcp_id FROM prescriptions) p ON h.hcp_id = p.hcp_id;

-- View feature distribution
SELECT 
    is_prescriber,
    COUNT(*) as hcp_count,
    ROUND(AVG(total_engagements_90d), 1) as avg_engagements,
    ROUND(AVG(avg_engagement_quality), 1) as avg_quality,
    ROUND(AVG(days_since_last_engagement), 0) as avg_days_since_contact
FROM hcp_features
GROUP BY is_prescriber
ORDER BY is_prescriber DESC;

---

## Paso 5: Train Prescribing Propensity Model

### Using SNOWFLAKE.ML.CLASSIFICATION

Train a model to predict `is_prescriber` (0 or 1) based on engagement features.

**Model will learn:**
- Which features predict prescribing
- Optimal thresholds for engagement
- Feature importance

In [ ]:
-- Train classification model for prescribing propensity
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION hcp_propensity_model(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'hcp_features'),
    TARGET_COLNAME => 'is_prescriber',
    CONFIG_OBJECT => {'evaluate': TRUE}
);

SELECT 'Model training complete!' as status;

---

## Paso 6: Evaluar Rendimiento del Modelo

View model metrics to understand prediction quality

In [ ]:
-- View model evaluation metrics
CALL hcp_propensity_model!SHOW_EVALUATION_METRICS();

-- Ver importancia de variables (which features matter most)
CALL hcp_propensity_model!SHOW_FEATURE_IMPORTANCE();

-- Ver matriz de confusión
CALL hcp_propensity_model!SHOW_CONFUSION_MATRIX();

---

## Paso 7: Score All HCPs with Propensity

### Using !PREDICT()

Generar propensity scores (0-100%) for all HCPs to prioritize targeting

In [ ]:
-- Score all HCPs with prescribing propensity
CREATE OR REPLACE TABLE hcp_propensity_scores AS
SELECT 
    hcp_id,
    specialty,
    years_in_practice,
    patient_volume_score,
    total_engagements_90d,
    avg_engagement_quality,
    days_since_last_engagement,
    is_prescriber as actual_prescriber,
    -- Predict propensity
    hcp_propensity_model!PREDICT(
        INPUT_DATA => OBJECT_CONSTRUCT(*)
    )['class']::INTEGER as predicted_prescriber,
    hcp_propensity_model!PREDICT(
        INPUT_DATA => OBJECT_CONSTRUCT(*)
    )['probability']['1']::FLOAT * 100 as propensity_score,
    -- Categorize propensity
    CASE 
        WHEN propensity_score >= 70 THEN '🔥 Hot Lead'
        WHEN propensity_score >= 50 THEN '🟡 Warm Lead'
        WHEN propensity_score >= 30 THEN '🟢 Cold Lead'
        ELSE '❄️ Very Low'
    END as propensity_category,
    -- Recommended action
    CASE 
        WHEN propensity_score >= 70 AND days_since_last_engagement > 14 
            THEN 'Schedule rep visit within 7 days'
        WHEN propensity_score >= 50 AND days_since_last_engagement > 30 
            THEN 'Send clinical update email + schedule visit'
        WHEN propensity_score >= 30 
            THEN 'Invite to upcoming event'
        ELSE 'Maintain nurture cadence'
    END as recommended_action
FROM hcp_features;

-- View top propensity HCPs
SELECT 
    hcp_id,
    specialty,
    total_engagements_90d as engagements_90d,
    ROUND(propensity_score, 1) as propensity_pct,
    propensity_category,
    recommended_action
FROM hcp_propensity_scores
ORDER BY propensity_score DESC
LIMIT 20;

---

## Paso 8: Generar Rep Target Lists

Create prioritized call lists for sales reps

In [ ]:
-- Generar rep target lists (prioritized by propensity)
CREATE OR REPLACE VIEW rep_target_lists AS
WITH rep_assignments AS (
    SELECT DISTINCT 
        e.rep_id,
        e.hcp_id
    FROM engagements e
    WHERE e.engagement_date >= DATEADD('day', -180, CURRENT_DATE())
),
hcp_priorities AS (
    SELECT 
        ra.rep_id,
        hps.hcp_id,
        hps.specialty,
        hps.propensity_score,
        hps.propensity_category,
        hps.recommended_action,
        hps.days_since_last_engagement,
        hps.total_engagements_90d,
        ROW_NUMBER() OVER (PARTITION BY ra.rep_id ORDER BY hps.propensity_score DESC) as priority_rank
    FROM rep_assignments ra
    JOIN hcp_propensity_scores hps ON ra.hcp_id = hps.hcp_id
    WHERE hps.propensity_score >= 30  -- Only actionable HCPs
)
SELECT * FROM hcp_priorities
WHERE priority_rank <= 20;  -- Top 20 per rep

-- View sample target list
SELECT 
    rep_id,
    COUNT(*) as total_targets,
    COUNT(CASE WHEN propensity_category = '🔥 Hot Lead' THEN 1 END) as hot_leads,
    COUNT(CASE WHEN propensity_category = '🟡 Warm Lead' THEN 1 END) as warm_leads,
    ROUND(AVG(propensity_score), 1) as avg_propensity
FROM rep_target_lists
GROUP BY rep_id
ORDER BY avg_propensity DESC
LIMIT 10;

---

## Paso 9: Interactive HCP Engagement Dashboard

Visualize propensity scores, target lists, and recommendations

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🎯 HCP Engagement Optimizer")
st.markdown("### Prescribing propensity & next-best-actions")

# Key metrics
total_hcps = session.sql("SELECT COUNT(*) as cnt FROM hcp_propensity_scores").collect()[0]['CNT']
hot_leads = session.sql("SELECT COUNT(*) as cnt FROM hcp_propensity_scores WHERE propensity_category='🔥 Hot Lead'").collect()[0]['CNT']
prescribers = session.sql("SELECT COUNT(*) as cnt FROM hcp_propensity_scores WHERE actual_prescriber=1").collect()[0]['CNT']
avg_propensity = session.sql("SELECT ROUND(AVG(propensity_score),1) as avg FROM hcp_propensity_scores").collect()[0]['AVG']

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total HCPs", int(total_hcps))
with col2:
    st.metric("Hot Leads", int(hot_leads))
with col3:
    st.metric("Current Prescribers", int(prescribers))
with col4:
    st.metric("Avg Propensity", f"{avg_propensity}%")

# Propensity distribution
st.markdown("---")
st.subheader("📊 Propensity Score Distribution")

col1, col2 = st.columns(2)

with col1:
    dist = session.sql("""
        SELECT propensity_category, COUNT(*) as count
        FROM hcp_propensity_scores
        GROUP BY propensity_category
        ORDER BY 
            CASE propensity_category
                WHEN '🔥 Hot Lead' THEN 1
                WHEN '🟡 Warm Lead' THEN 2
                WHEN '🟢 Cold Lead' THEN 3
                ELSE 4
            END
    """).to_pandas()
    st.bar_chart(dist.set_index('PROPENSITY_CATEGORY')['COUNT'])

with col2:
    by_specialty = session.sql("""
        SELECT specialty, ROUND(AVG(propensity_score),1) as avg_score
        FROM hcp_propensity_scores
        GROUP BY specialty
        ORDER BY avg_score DESC
    """).to_pandas()
    st.bar_chart(by_specialty.set_index('SPECIALTY')['AVG_SCORE'])

# Top opportunities
st.markdown("---")
st.subheader("🔥 Top Prescribing Opportunities")

top_opps = session.sql("""
    SELECT hcp_id, specialty, total_engagements_90d, 
           ROUND(propensity_score,1) as propensity, 
           propensity_category, recommended_action
    FROM hcp_propensity_scores
    WHERE propensity_score >= 50
    ORDER BY propensity_score DESC
    LIMIT 20
""").to_pandas()

st.dataframe(top_opps, use_container_width=True, hide_index=True)

# Model performance
st.markdown("---")
st.subheader("🎯 Model Performance")

try:
    # Get evaluation metrics from trained model
    metrics_result = session.sql("CALL hcp_propensity_model!SHOW_EVALUATION_METRICS()").collect()
    
    if len(metrics_result) > 0:
        # Convert to DataFrame to inspect structure
        metrics_df_raw = pd.DataFrame([row.as_dict() for row in metrics_result])
        
        # Try to extract metrics - handle different possible column names
        metrics_data = []
        
        # Check what columns are actually present
        if 'METRIC' in metrics_df_raw.columns and 'VALUE' in metrics_df_raw.columns:
            # Standard format
            for _, row in metrics_df_raw.iterrows():
                metric_name = str(row['METRIC'])
                metric_value = row['VALUE']
                
                # Format the metric based on type
                if any(term in metric_name.upper() for term in ['ACCURACY', 'PRECISION', 'RECALL', 'F1']):
                    formatted_value = f"{float(metric_value):.1%}" if metric_value is not None else "N/A"
                else:
                    formatted_value = f"{float(metric_value):.3f}" if metric_value is not None else "N/A"
                
                metrics_data.append({
                    "Metric": metric_name,
                    "Value": formatted_value
                })
        else:
            # Try alternate format - display first few columns
            for _, row in metrics_df_raw.iterrows():
                row_dict = row.to_dict()
                # Take first two non-null values as metric name and value
                values = [v for v in row_dict.values() if v is not None]
                if len(values) >= 2:
                    metrics_data.append({
                        "Metric": str(values[0]),
                        "Value": str(values[1])
                    })
        
        if metrics_data:
            metrics_display_df = pd.DataFrame(metrics_data)
            st.table(metrics_display_df)
        else:
            st.info("Model trained successfully. Evaluation metrics format not recognized.")
            # Show raw data for debugging
            with st.expander("View raw metrics data"):
                st.dataframe(metrics_df_raw)
    else:
        st.info("No metrics returned. Model may need retraining with evaluate=TRUE.")
        
except Exception as e:
    st.warning(f"Error retrieving metrics: {str(e)[:200]}")
    st.info("This may occur if the model hasn't been trained yet. Run cells 10-12 to train the model.")

# Rep target lists
st.markdown("---")
st.subheader("📋 Rep Target Lists Summary")

rep_summary = session.sql("""
    SELECT rep_id, COUNT(*) as targets, 
           COUNT(CASE WHEN propensity_category='🔥 Hot Lead' THEN 1 END) as hot_leads,
           ROUND(AVG(propensity_score),1) as avg_propensity
    FROM rep_target_lists
    GROUP BY rep_id
    ORDER BY avg_propensity DESC
    LIMIT 15
""").to_pandas()

st.dataframe(rep_summary, use_container_width=True, hide_index=True)

# Engagement vs. propensity
st.markdown("---")
st.subheader("📈 Engagement Impact on Propensity")

engagement_impact = session.sql("""
    SELECT 
        CASE 
            WHEN total_engagements_90d = 0 THEN '0'
            WHEN total_engagements_90d <= 3 THEN '1-3'
            WHEN total_engagements_90d <= 6 THEN '4-6'
            WHEN total_engagements_90d <= 10 THEN '7-10'
            ELSE '11+'
        END as engagement_bucket,
        ROUND(AVG(propensity_score),1) as avg_propensity
    FROM hcp_propensity_scores
    GROUP BY engagement_bucket
    ORDER BY 
        CASE engagement_bucket
            WHEN '0' THEN 1
            WHEN '1-3' THEN 2
            WHEN '4-6' THEN 3
            WHEN '7-10' THEN 4
            ELSE 5
        END
""").to_pandas()

st.line_chart(engagement_impact.set_index('ENGAGEMENT_BUCKET')['AVG_PROPENSITY'])
st.caption("Higher engagement frequency correlates with higher prescribing propensity")

# Download
st.markdown("---")
csv = top_opps.to_csv(index=False)
st.download_button("📥 Download Top Opportunities", data=csv, file_name="hcp_opportunities.csv", mime="text/csv")

---

## ✅ Tutorial Complete!

### What You've Learned

1. ✅ Used **SNOWFLAKE.ML.CLASSIFICATION** for propensity modeling
2. ✅ Engineered features from engagement history
3. ✅ Predicted HCP prescribing behavior
4. ✅ Scored propensity with `!PREDICT()`
5. ✅ Generated prioritized rep target lists
6. ✅ Built interactive engagement dashboards

### Key ML Techniques

**Ingeniería de Variables:**
- Engagement frequency (90-day window)
- Recency (days since last touch)
- Channel diversity
- Engagement quality scores
- Demographic factors

**Model Training:**
```sql
CREATE SNOWFLAKE.ML.CLASSIFICATION model_name(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'features_table'),
    TARGET_COLNAME => 'is_prescriber',
    CONFIG_OBJECT => {'evaluate': TRUE}
);
```

**Scoring:**
```sql
model!PREDICT(INPUT_DATA => OBJECT_CONSTRUCT(*))['probability']['1']::FLOAT
```

### Impacto de Negocio

- **Rep productivity**: Focus on high-propensity HCPs
- **Resource optimization**: Allocate budget to effective channels
- **Personalization**: Tailor outreach based on propensity level
- **Conversion improvement**: 20-40% lift in prescribing rates

### Next Steps

- Add real-time propensity updates as engagements occur
- Incorporate external data (NPI, claims, affiliations)
- Build next-best-action recommenders
- Create automated rep alerts for hot leads
- Track propensity score changes over time

### Resources

- [Snowflake ML CLASSIFICATION](https://docs.snowflake.com/en/user-guide/ml-functions/classification)
- [ML Functions](https://docs.snowflake.com/en/user-guide/ml-functions)

---

**Congratulations!** You've built a complete HCP engagement optimization system.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `HCP_ENGAGEMENT_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data and models will be permanently deleted.


In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS HCP_ENGAGEMENT_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;